In [ ]:
import numpy as np
from qiskit import Aer
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.algorithms.minimum_eigensolvers import VQE
from qiskit.algorithms.optimizers import COBYLA
from qiskit.primitives import Estimator

In [ ]:
# Assumptions

N = 4                      # interior grid (4x4 = 16 unknowns)
h = 1.0 / (N + 1)
dim = N * N

def idx(i, j):
    return i * N + j

A = np.zeros((dim, dim))

for i in range(N):
    for j in range(N):
        k = idx(i, j)
        A[k, k] = -4
        if i > 0:     A[k, idx(i-1, j)] = 1
        if i < N-1:   A[k, idx(i+1, j)] = 1
        if j > 0:     A[k, idx(i, j-1)] = 1
        if j < N-1:   A[k, idx(i, j+1)] = 1

A /= h**2

In [ ]:
b = np.zeros(dim)
for j in range(N):
    b[idx(N-1, j)] = -1.0 / h   # moving lid (top boundary)

b = b / np.linalg.norm(b)

In [ ]:
n_qubits = int(np.ceil(np.log2(dim)))
size = 2**n_qubits

A_pad = np.zeros((size, size))
A_pad[:dim, :dim] = A

# Decompose into Pauli operators
H = SparsePauliOp.from_operator(A_pad)

In [ ]:
ansatz = TwoLocal(
    n_qubits,
    rotation_blocks=["ry", "rz"],
    entanglement_blocks="cz",
    entanglement="linear",
    reps=2
)

optimizer = COBYLA(maxiter=300)
estimator = Estimator()

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer
)


In [ ]:
result = vqe.compute_minimum_eigenvalue(H)

state = Statevector.from_instruction(
    ansatz.assign_parameters(result.optimal_parameters)
)

psi = np.real(state.data[:dim])
psi /= np.linalg.norm(psi)

In [ ]:
psi_physical = np.linalg.solve(A, b[:dim])

print("VQA solution (normalized):")
print(psi.reshape(N, N))

print("\nClassical solution:")
print(psi_physical.reshape(N, N))